# ARS Fresh Fixed-Steering RFT and Intensity Study

This notebook starts a new experiment from the frozen Phi-2 backbone. It does not search for or load prior rollouts, norm files, or SteerNet checkpoints. It trains the fixed V2 route `[12, 15, 18, 21]`, calibrates post-cap steering intensity on held-out GSM8K validation questions, and evaluates baseline, unit steering, and the selected intensity on the same deterministic GSM8K test subset.

The notebook records phase failures and continues to the final status cell. A dependent phase is skipped when its required artifact was not created.


## 1. Kaggle runtime and repository checkout

Enable Dual T4 in Kaggle settings. This cell checks out the current `development` branch and streams subprocess progress into cell output.


In [ ]:
import codecs
import json
import os
import selectors
import subprocess
import sys
import time
from pathlib import Path

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"

REPO_URL = "https://github.com/Abdullah182155/adaptive-residual-steering.git"
REPO_DIR = Path("/kaggle/working/adaptive-residual-steering")
BRANCH = "development"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print(f"Repository: {REPO_DIR} ({BRANCH})", flush=True)

def _emit_stream_line(state, line, *, force=False):
    """Print child output snapshots as normal notebook lines with a short throttle."""
    now = time.monotonic()
    if line and (force or now - state["last_update"] >= 1.0 or "100%" in line):
        print(f"{state['prefix']}{line}", flush=True)
        state["last_update"] = now


def _consume_stream_chunk(state, chunk):
    """Split child output on newline and tqdm carriage-return boundaries."""
    state["pending"] += state["decoder"].decode(chunk)
    while "\r" in state["pending"] or "\n" in state["pending"]:
        cr = state["pending"].find("\r") if "\r" in state["pending"] else len(state["pending"])
        lf = state["pending"].find("\n") if "\n" in state["pending"] else len(state["pending"])
        boundary = min(cr, lf)
        separator = state["pending"][boundary]
        line, state["pending"] = state["pending"][:boundary], state["pending"][boundary + 1:]
        _emit_stream_line(state, line, force=(separator == "\n" or "100%" in line))


def _finish_stream(state):
    state["pending"] += state["decoder"].decode(b"", final=True)
    _emit_stream_line(state, state["pending"], force=True)


def _start_live_process(command, env=None):
    return subprocess.Popen(
        command,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=0,
    )


def run_live(command, *, env=None, prefix=""):
    """Run one child and render its tqdm refreshes as flushed notebook output."""
    run_parallel_live([(prefix, command, env)])


def run_parallel_live(jobs):
    """Run child commands concurrently and stream each labelled pipe in this thread."""
    selector = selectors.DefaultSelector()
    processes = []
    for prefix, command, env in jobs:
        process = _start_live_process(command, env)
        processes.append((prefix, command, process))
        state = {
            "prefix": prefix,
            "decoder": codecs.getincrementaldecoder("utf-8")(errors="replace"),
            "pending": "",
            "last_update": 0.0,
        }
        selector.register(process.stdout, selectors.EVENT_READ, state)

    while selector.get_map():
        for key, _ in selector.select(timeout=0.25):
            chunk = os.read(key.fileobj.fileno(), 4096)
            if chunk:
                _consume_stream_chunk(key.data, chunk)
            else:
                _finish_stream(key.data)
                selector.unregister(key.fileobj)
                key.fileobj.close()

    selector.close()
    failures = []
    for prefix, command, process in processes:
        return_code = process.wait()
        if return_code:
            failures.append(f"{prefix.strip()} exited with code {return_code}: {command}")
    if failures:
        raise RuntimeError("\n".join(failures))


## 2. Fresh-run configuration

`SMOKE_TEST=True` uses small volumes for an end-to-end check. Production uses fresh rollouts and an 800-question paired GSM8K test sample. The fixed route, frozen backbone, `phase1_lr=1.5e-4`, identity magnitude, and clean NLL objective are locked.


In [ ]:
import json
from datetime import datetime
from pathlib import Path
import os
import sys

SMOKE_TEST = False
ATTENTION_IMPLEMENTATION = "sdpa"
os.environ["ARS_ATTENTION_IMPLEMENTATION"] = ATTENTION_IMPLEMENTATION
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = Path("report/fixed_steering_fresh") / RUN_ID
RFT_DIR = RUN_DIR / "rft"
NORM_FILE = RUN_DIR / "norms_v2_fresh.json"
CHECKPOINT = RUN_DIR / "steernet_v2_fresh.pt"
BENCH_RECORD = RUN_DIR / "gsm8k_paired_record.json"

FIXED_ROUTE = [12, 15, 18, 21]
SEED = 42
if SMOKE_TEST:
    CALIB_N, EQUIVALENCE_N, GATE0_N = 24, 5, 5
    COLLECTION_QUESTIONS, COLLECTION_K = 40, 2
    TRAIN_EPOCHS, BENCH_N_TEST, ALPHA_CALIB_N = 1, 30, 24
else:
    CALIB_N, EQUIVALENCE_N, GATE0_N = 128, 100, 60
    COLLECTION_QUESTIONS, COLLECTION_K = 2000, 8
    TRAIN_EPOCHS, BENCH_N_TEST, ALPHA_CALIB_N = 2, 800, 100
ALPHA_GRID = [0.5, 0.75, 1.0, 1.25]
TRAIN_SHOT_WEIGHTS = [0.50, 0.30, 0.20]  # 0/2/4-shot rows
BENCH_SHOTS = [0, 2]
PHASE_STATUS = {name: "pending" for name in ("runtime", "tests", "norms", "equivalence", "gate0", "collection", "training", "alpha_calibration", "benchmark")}
RUN_DIR.mkdir(parents=True, exist_ok=False)
RFT_DIR.mkdir(parents=True, exist_ok=True)
print({"run_dir": str(RUN_DIR), "fresh_artifacts_only": True, "route": FIXED_ROUTE, "calibration_questions": CALIB_N, "collection_questions": COLLECTION_QUESTIONS, "rollouts_per_question": COLLECTION_K, "epochs": TRAIN_EPOCHS, "micro_batch": 1, "grad_accum": 32, "alpha_grid": ALPHA_GRID, "benchmark_n": BENCH_N_TEST, "shots": BENCH_SHOTS}, flush=True)


## 3. Runtime and tests

Confirm GPU visibility and run the repository test suite. Failures are recorded so the notebook can report the complete run state.


In [ ]:
try:
    gpu_query = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True, check=True)
    visible_gpus = [line.strip() for line in gpu_query.stdout.splitlines() if line.strip()]
    print("Visible GPUs:", visible_gpus, flush=True)
    PHASE_STATUS["runtime"] = "passed" if len(visible_gpus) >= 2 else "warning_single_gpu"
except Exception as exc:
    visible_gpus = []
    PHASE_STATUS["runtime"] = f"diagnostic_failed: {type(exc).__name__}: {exc}"

try:
    test_result = subprocess.run([sys.executable, "-u", "-m", "pytest", "tests/", "-x", "-q"], check=False)
    PHASE_STATUS["tests"] = "passed" if test_result.returncode == 0 else f"failed_exit_{test_result.returncode}"
except Exception as exc:
    PHASE_STATUS["tests"] = f"failed: {type(exc).__name__}: {exc}"
print("Test status:", PHASE_STATUS["tests"], flush=True)


## 4. Calibrate fresh V2 layer norms

Calibration reads GSM8K training questions only and writes a new run-specific p99 file. No uploaded calibration is consulted.


In [ ]:
try:
    run_live([sys.executable, "-u", "scripts/calibrate_norms.py", "--n_samples", str(CALIB_N), "--seed", str(SEED), "--candidate_layers", *map(str, FIXED_ROUTE), "--max_length", "768", "--output", str(NORM_FILE)], prefix="[norms] ")
    PHASE_STATUS["norms"] = "passed" if NORM_FILE.is_file() else "failed_missing_output"
except Exception as exc:
    PHASE_STATUS["norms"] = f"failed: {type(exc).__name__}: {exc}"
    print("Fresh norm calibration failed; later dependent stages will record skips.", flush=True)


## 5. Equivalence and Gate 0 diagnostics

Batched generation equivalence is diagnostic for this run. Gate 0 records sampled-vs-greedy quality but is nonblocking so a small or noisy probe does not terminate the notebook.


In [ ]:
try:
    run_live([sys.executable, "-u", "scripts/run_rft_pipeline.py", "--stage", "equivalence", "--steernet_arch", "v2", "--norm_calibration", str(NORM_FILE), "--n_questions", str(EQUIVALENCE_N), "--output_dir", str(RFT_DIR)], prefix="[equivalence] ")
    PHASE_STATUS["equivalence"] = "completed_diagnostic"
except Exception as exc:
    PHASE_STATUS["equivalence"] = f"diagnostic_failed: {type(exc).__name__}: {exc}"
    print("Equivalence diagnostic did not pass; continuing with batch-1 training/evaluation paths.", flush=True)

if NORM_FILE.is_file():
    try:
        run_live([sys.executable, "-u", "scripts/run_rft_pipeline.py", "--stage", "gate0", "--steernet_arch", "v2", "--norm_calibration", str(NORM_FILE), "--n_questions", str(GATE0_N), "--k", "8", "--temperature", "0.7", "--gate0_nonblocking", "--output_dir", str(RFT_DIR)], prefix="[gate0] ")
        PHASE_STATUS["gate0"] = "completed_diagnostic"
    except Exception as exc:
        PHASE_STATUS["gate0"] = f"failed: {type(exc).__name__}: {exc}"
        print("Gate 0 failed; rollout workers will still be attempted and their outcome recorded.", flush=True)
else:
    PHASE_STATUS["gate0"] = "skipped_missing_fresh_norms"


## 6. Collect new positive rollouts on both T4s

Both workers sample disjoint portions of one deterministic question set. Each shard writes new JSONL and stats into this run directory. Low yield is recorded rather than raised; training separately reports if no usable positives exist.


In [ ]:
if NORM_FILE.is_file():
    collection_env = os.environ.copy()
    jobs = []
    for shard in (0, 1):
        gpu = str(shard) if len(visible_gpus) >= 2 else "0"
        env = collection_env.copy()
        env["CUDA_VISIBLE_DEVICES"] = gpu
        command = [sys.executable, "-u", "scripts/run_rft_pipeline.py", "--stage", "collect", "--steernet_arch", "v2", "--norm_calibration", str(NORM_FILE), "--n_questions", str(COLLECTION_QUESTIONS), "--k", str(COLLECTION_K), "--temperature", "0.7", "--n_shards", "2", "--shard", str(shard), "--allow_low_yield", "--output_dir", str(RFT_DIR)]
        jobs.append((f"[GPU{gpu}/shard{shard}] ", command, env))
    try:
        if len(visible_gpus) >= 2:
            print("Collecting rollout shards in parallel on two GPUs.", flush=True)
            run_parallel_live(jobs)
        else:
            print("One GPU detected; collecting both rollout shards sequentially on GPU 0.", flush=True)
            for job in jobs:
                run_live(job[1], env=job[2], prefix=job[0])
        shards = [RFT_DIR / f"shard{i}.jsonl" for i in (0, 1)]
        PHASE_STATUS["collection"] = "completed" if all(p.is_file() and p.stat().st_size for p in shards) else "incomplete_outputs"
    except Exception as exc:
        existing = [p.name for p in RFT_DIR.glob("shard*.jsonl") if p.is_file() and p.stat().st_size]
        PHASE_STATUS["collection"] = f"partial_or_failed: {type(exc).__name__}: {exc}; available={existing}"
else:
    PHASE_STATUS["collection"] = "skipped_missing_fresh_norms"


## 7. Train a fresh fixed-route SteerNet

Training uses only the shards above, route `[12, 15, 18, 21]`, rank 16, p99 norm caps, `m=1`, positive-completion token-mean NLL, a balanced 0/2/4-shot mix, and three meaning-equivalent instruction templates. Micro-batch 1 with accumulation 32 reduces T4 activation peaks while preserving the effective batch size.


In [ ]:
shards_ready = any(p.is_file() and p.stat().st_size for p in RFT_DIR.glob("shard*.jsonl"))
if NORM_FILE.is_file() and shards_ready:
    train_env = os.environ.copy()
    train_env["CUDA_VISIBLE_DEVICES"] = "0"
    try:
        run_live([sys.executable, "-u", "scripts/run_rft_pipeline.py", "--stage", "train", "--steernet_arch", "v2", "--norm_calibration", str(NORM_FILE), "--epochs", str(TRAIN_EPOCHS), "--batch_size", "1", "--grad_accum", "32", "--checkpoint", str(CHECKPOINT), "--output_dir", str(RFT_DIR), "--train_shot_weights", *map(str, TRAIN_SHOT_WEIGHTS), "--diversify_prompt_templates"], env=train_env, prefix="[train] ")
        PHASE_STATUS["training"] = "passed" if CHECKPOINT.is_file() else "failed_missing_checkpoint"
    except Exception as exc:
        PHASE_STATUS["training"] = f"failed: {type(exc).__name__}: {exc}"
        print("Training failed; intensity calibration and test evaluation will be skipped.", flush=True)
else:
    PHASE_STATUS["training"] = "skipped_missing_fresh_norms_or_rollouts"


## 8. Calibrate steering intensity on held-out validation questions

The trainer selects epochs on the first validation slice. This sweep uses a disjoint slice, compares post-cap intensities for both 0-shot and 2-shot, and writes its choices. The final 800-question test subset is not used to choose alpha.


In [ ]:
ALPHA_REPORT = RFT_DIR / "alpha_calibration.json"
if CHECKPOINT.is_file() and NORM_FILE.is_file():
    try:
        run_live([sys.executable, "-u", "scripts/run_rft_pipeline.py", "--stage", "alpha_sweep", "--steernet_arch", "v2", "--norm_calibration", str(NORM_FILE), "--checkpoint", str(CHECKPOINT), "--n_questions", str(ALPHA_CALIB_N), "--alpha_values", *map(str, ALPHA_GRID), "--output_dir", str(RFT_DIR)], prefix="[alpha] ")
        PHASE_STATUS["alpha_calibration"] = "passed" if ALPHA_REPORT.is_file() else "failed_missing_report"
    except Exception as exc:
        PHASE_STATUS["alpha_calibration"] = f"failed: {type(exc).__name__}: {exc}"
        print("Intensity calibration failed; final test evaluation uses alpha=1.0.", flush=True)
else:
    PHASE_STATUS["alpha_calibration"] = "skipped_missing_checkpoint_or_norms"

SELECTED_ALPHA = 1.0
ALPHA_SELECTION = {"method": "0.7 * 0-shot paired delta + 0.3 * 2-shot paired delta"}
if ALPHA_REPORT.is_file():
    alpha_data = json.loads(ALPHA_REPORT.read_text(encoding="utf-8"))
    alpha_scores = {}
    for shot, values in alpha_data["shots"].items():
        for item in values["candidates"]:
            alpha_scores.setdefault(float(item["alpha"]), {})[str(shot)] = float(item["paired_delta_lenient_pp"])
    eligible = [alpha for alpha, scores in alpha_scores.items() if "0" in scores and "2" in scores]
    if eligible:
        SELECTED_ALPHA = max(eligible, key=lambda alpha: (0.7 * alpha_scores[alpha]["0"] + 0.3 * alpha_scores[alpha]["2"], -abs(alpha - 1.0)))
        ALPHA_SELECTION["validation_paired_deltas"] = {str(alpha): alpha_scores[alpha] for alpha in eligible}
        ALPHA_SELECTION["selected_alpha"] = SELECTED_ALPHA
print({"alpha_selection": ALPHA_SELECTION}, flush=True)


## 9. Paired GSM8K test benchmark

All arms use the same seeded test questions and prompt-template schedule. The selected alpha is reported alongside the unit-intensity fixed arm.


In [ ]:
if CHECKPOINT.is_file() and NORM_FILE.is_file():
    bench_env = os.environ.copy()
    bench_env["CUDA_VISIBLE_DEVICES"] = "0"
    try:
        run_live([sys.executable, "-u", "scripts/run_ars_pipeline.py", "--stage", "bench", "--steernet_arch", "v2", "--n_test", str(BENCH_N_TEST), "--test_seed", str(SEED), "--shots", *map(str, BENCH_SHOTS), "--arms", f"baseline,steer_only,steer_gain:{SELECTED_ALPHA}", "--record", str(BENCH_RECORD), "--report", "--set", f"layer_norms_calibration={NORM_FILE}", "--set", f"lora_save_path={CHECKPOINT}", "--set", f"report_dir={RUN_DIR / 'benchmark'}", "--set", "diversify_prompts=true", "--set", "use_gate=false"], env=bench_env, prefix="[benchmark] ")
        PHASE_STATUS["benchmark"] = "passed" if BENCH_RECORD.is_file() else "failed_missing_record"
    except Exception as exc:
        PHASE_STATUS["benchmark"] = f"failed: {type(exc).__name__}: {exc}"
else:
    PHASE_STATUS["benchmark"] = "skipped_missing_checkpoint_or_norms"


## 10. Final run status and artifacts

This summary is the copyable run receipt. Check the paired deltas and confidence intervals before treating the 800-question result as final.


In [ ]:
summary = {
    "run_id": RUN_ID,
    "run_dir": str(RUN_DIR),
    "phase_status": PHASE_STATUS,
    "artifacts": {
        "fresh_norms": str(NORM_FILE) if NORM_FILE.is_file() else None,
        "fresh_rollout_shards": [str(p) for p in sorted(RFT_DIR.glob("shard*.jsonl")) if p.is_file()],
        "fresh_checkpoint": str(CHECKPOINT) if CHECKPOINT.is_file() else None,
        "alpha_calibration": str(ALPHA_REPORT) if ALPHA_REPORT.is_file() else None,
        "benchmark_record": str(BENCH_RECORD) if BENCH_RECORD.is_file() else None,
    },
    "route": FIXED_ROUTE,
    "selected_alpha": SELECTED_ALPHA,
    "test_n": BENCH_N_TEST,
    "shots": BENCH_SHOTS,
}
print(json.dumps(summary, indent=2), flush=True)


## Interpretation guardrails

- Compare paired per-question outcomes; do not compare unpaired percentages across runs.
- Treat the 800-question sample as an interim estimate, with its interval, not as the full GSM8K test result.
- Keep 0-shot as the primary target and inspect 2-shot separately for regressions.
- Alpha calibration is global per run in this first pass. Per-question alpha is a follow-up only if the held-out sweep shows a repeatable advantage and enough examples to fit and validate it without test leakage.
